# F-12: Bidirectional (lead) soil lags for pooled RFm gap-filling

RFm's champion config (D-35/D-49) only ever uses **backward-only** soil lag features
(`swc_l{lag}`/`ts_l{lag}` via `.shift(lag)`, `LAG_HOURS=[168,336,504,672]`) — despite the fact
that (a) the upstream met/soil driver gap-filling (`reddyproc_pipeline.py`) already uses a
bidirectional, centered expanding window, and (b) F-11's SAITS alternative is inherently
bidirectional (unmasked self-attention). This has never been tested for RFm itself. This notebook
runs a controlled 3-arm ablation on *only* the swc/ts lag block, everything else held fixed
(partial pooling T2+T4+T9 + tower dummies, EXT sourcing, full-period gap-CV):

- **Arm A (baseline)** — current backward-only lags, rerun fresh here for a clean same-run
  comparison point (not just cited from `BEST_RESULTS.md`, since prior reruns of this exact
  protocol have shown small cross-run drift, e.g. F-08 vs F-09a).
- **Arm B (bidir)** — Arm A's features **plus** new forward lags `swc_f{lag}`/`ts_f{lag}` via
  `.shift(-lag)`, same `LAG_HOURS`. Tests: does adding future context help at all?
- **Arm C (leadonly)** — forward lags **replacing** the backward ones, same feature *count* as
  Arm A. Tests: is it *direction* that matters, or just having more temporal context of any kind?

Champion to beat (median R² across 5 gap scenarios): T2=0.574, T4=0.402, T9=0.418
(`results/f09a_summary.csv`, `BEST_RESULTS.md` §1).

**Phase 1 (this section): smoke test.** Tower 4, scenario `m` (32h), 1 rep, all 3 arms — GO/NO-GO
gate + per-fit wall-time estimate before committing to the full run.

In [1]:
from pathlib import Path
import sys, time, datetime

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score

sys.path.insert(0, str(Path("../../src/models").resolve()))
from gapfill_rfm import load_ext, cfg, ts_col_for, feat_list, frame as frame_baseline, \
    fit, TOWERS, LAG_HOURS, DUM  # noqa: E402  single source of truth, never edited

sys.path.insert(0, str(Path("../../src/interpretability").resolve()))
from importance import native_importance_tree  # noqa: E402

HOURLY = Path("../../data/Hourly")
RESULTS = Path("../../results")

N_REPS, MASK_FRAC = 2, 0.25
SCENARIOS = {"vs": 1, "s": 4, "m": 32, "l": 288, "m1": "mixed"}
DOMAIN = {2: ("2017-10-01", "2019-06-30"), 4: ("2017-10-01", "2023-12-31"), 9: ("2020-02-01", "2023-12-31")}

**Reps note:** `N_REPS=2` (not F-08's 5) — a documented reduction, matching F-09a's own precedent for this exact protocol under time pressure. Triggered here because the first attempt at `N_REPS=5` (225 fits, ~3.45h extrapolated from the smoke test) was killed by the environment after ~2h20m with zero progress saved (`nbconvert --inplace` only writes the file once the whole run finishes). At `N_REPS=2`, the full run is 3 arms x 3 towers x 5 scenarios x 2 reps = 90 fits (~83 min extrapolated), executed cell-by-cell (one cell per arm x tower pair) via a checkpointing driver that writes the notebook to disk after every cell — worst-case loss on another kill is one (arm, tower) pair, not the whole run. Coverage (3 arms x 3 towers x 5 scenarios) is unchanged; only the rep count is reduced, per the plan's explicit rule that reps is the one approved dial, never towers/scenarios/arms.

## Shared gap-CV harness (duplicated per this repo's established per-notebook convention, F07/F08/F11 — not centralized)

In [2]:
def insert_calendar_gaps(df_qc, target, domain_mask, gap_hours, n_reps=N_REPS, seed=0):
    dom_ts = df_qc.index[domain_mask]; valid = df_qc.loc[domain_mask, target].notna().values
    n = len(dom_ts); target_n = max(1, int(valid.sum() * MASK_FRAC)); rb = np.random.default_rng(seed); reps = []
    for _ in range(n_reps):
        rng = np.random.default_rng(int(rb.integers(0, 2**31))); occ = np.zeros(n, bool); m = 0
        for sp in rng.permutation(n):
            if m >= target_n: break
            gh = int(rng.choice([1, 4, 32, 288])) if gap_hours == "mixed" else gap_hours
            ep = min(int(sp) + gh, n)
            if occ[sp:ep].any(): continue
            occ[sp:ep] = True; m += int(valid[sp:ep].sum())
        reps.append(dom_ts[occ & valid])
    return reps


def dom_mask(idx, t):
    a, b = DOMAIN[t]
    return (idx >= pd.Timestamp(a)) & (idx <= pd.Timestamp(b))


def mets(y, p):
    y = np.asarray(y, float); p = np.asarray(p, float)
    r2 = r2_score(y, p) if np.var(y) > 0 else np.nan
    return r2, float(np.sqrt(np.mean((p - y) ** 2))), float(np.mean(np.abs(p - y))), float(np.mean(p - y))


def med_metrics(rows):
    if not rows: return {k: np.nan for k in ["R2", "RMSE", "MAE", "MBE"]}
    a = np.array(rows, float)
    return {"R2": np.nanmedian(a[:, 0]), "RMSE": np.median(a[:, 1]), "MAE": np.median(a[:, 2]), "MBE": np.median(a[:, 3])}

## Arm feature frames (additive clones of `gapfill_rfm.frame` — only the swc/ts lag block differs; everything else is reused unchanged)

In [3]:
def frame_bidir(t, pooled, d):
    """Arm B: baseline frame + forward (lead) soil lags via .shift(-lag)."""
    g = frame_baseline(t, pooled, d)
    c = cfg(t, ts_col_for(t))
    swc = d[c["swc"] + "__f"] if (c["swc"] + "__f") in d.columns else d[c["swc"]]
    ts = d[c["ts"] + "__f"] if (c["ts"] + "__f") in d.columns else d[c["ts"]]
    for lag in LAG_HOURS:
        g[f"swc_f{lag}"] = swc.shift(-lag)
        g[f"ts_f{lag}"] = ts.shift(-lag)
    return g


def frame_leadonly(t, pooled, d):
    """Arm C: backward lags dropped, forward lags kept -- same feature count as Arm A."""
    g = frame_bidir(t, pooled, d)
    drop = [f"swc_l{l}" for l in LAG_HOURS] + [f"ts_l{l}" for l in LAG_HOURS]
    return g.drop(columns=drop)


BASE_FEAT = feat_list()
LEAD_COLS = [f"swc_f{l}" for l in LAG_HOURS] + [f"ts_f{l}" for l in LAG_HOURS]
LAG_COLS = [f"swc_l{l}" for l in LAG_HOURS] + [f"ts_l{l}" for l in LAG_HOURS]
BIDIR_FEAT = BASE_FEAT + LEAD_COLS
LEADONLY_FEAT = [c for c in BASE_FEAT if c not in LAG_COLS] + LEAD_COLS

ARMS = {
    "baseline": dict(frame_fn=frame_baseline, feat=BASE_FEAT, fs_tag="RFm_pool_EXT_baselinelag"),
    "bidir": dict(frame_fn=frame_bidir, feat=BIDIR_FEAT, fs_tag="RFm_pool_EXT_bidirlag"),
    "leadonly": dict(frame_fn=frame_leadonly, feat=LEADONLY_FEAT, fs_tag="RFm_pool_EXT_leadonly"),
}
print({a: len(v["feat"]) for a, v in ARMS.items()})

{'baseline': 30, 'bidir': 38, 'leadonly': 30}


## Load data (EXT sourcing, single source shared by all 3 arms)

In [4]:
DF = load_ext()
print(DF.shape)

(70153, 524)


## Phase 1 -- smoke test: Tower 4, scenario `m` (32h), 1 rep, all 3 arms

GO/NO-GO gate: confirm each arm's frame construction is sane (no unexpected NaN blowup from the new `.shift(-lag)` columns) and get a per-fit wall-time estimate. Uses full pooled training (this tower's held-in rows + the other 2 towers' full domain rows) to match `run_rf`'s real per-fit cost, not a cheaper solo approximation. `n_reps=1` passed explicitly -- unaffected by the `N_REPS=2` global used for the full run.

In [5]:
t0 = time.time()
T, SC = 4, "m"
smoke_times = {}
for arm, spec in ARMS.items():
    ta = time.time()
    feat = spec["feat"] + DUM
    frames = {tt: spec["frame_fn"](tt, True, DF) for tt in TOWERS}
    g = frames[T]; dm = dom_mask(g.index, T)
    gt = insert_calendar_gaps(g, "target", dm, SCENARIOS[SC], n_reps=1, seed=0)[0]
    base = g[dm & g["target"].notna().values]; trd = base.drop(index=gt, errors="ignore")
    others = [frames[tt][dom_mask(frames[tt].index, tt) & frames[tt]["target"].notna().values]
              for tt in TOWERS if tt != T]
    trd = pd.concat([trd] + others, ignore_index=True)
    rf, imp = fit(feat, trd)
    yp = rf.predict(imp.transform(g.loc[gt, feat].values))
    r2, rmse, mae, mbe = mets(g.loc[gt, "target"].values, yp)
    smoke_times[arm] = time.time() - ta
    lead_present = [c for c in LEAD_COLS if c in g.columns]
    nan_frac = g[lead_present].isna().mean().max() if lead_present else float("nan")
    print(f"{arm}: R2={r2:.3f} RMSE={rmse:.1f} MAE={mae:.1f} MBE={mbe:.1f} "
          f"| n_feat={len(feat)} | max_lead_nan_frac={nan_frac:.4f} | {smoke_times[arm]:.1f}s")

print(f"\ntotal smoke-test wall time: {time.time()-t0:.1f}s")
per_fit = float(np.mean(list(smoke_times.values())))
n_fits_full = 3 * 3 * 5 * N_REPS  # arms x towers x scenarios x reps
print(f"mean per-fit time: {per_fit:.1f}s -> extrapolated full run ({n_fits_full} fits): "
      f"{per_fit*n_fits_full/60:.1f} min ({per_fit*n_fits_full/3600:.2f} h)")

baseline: R2=0.371 RMSE=104.9 MAE=44.8 MBE=2.3 | n_feat=33 | max_lead_nan_frac=nan | 49.2s


bidir: R2=0.380 RMSE=104.1 MAE=44.7 MBE=2.4 | n_feat=41 | max_lead_nan_frac=0.0096 | 62.3s


leadonly: R2=0.383 RMSE=103.9 MAE=44.7 MBE=2.7 | n_feat=33 | max_lead_nan_frac=0.0096 | 52.1s

total smoke-test wall time: 163.7s
mean per-fit time: 54.6s -> extrapolated full run (90 fits): 81.8 min (1.36 h)


## Phase 1 verdict: GO

(Reproduced from the first attempt: 165.8s total for the 3-arm smoke test, R2 in the narrow 0.371-0.383 band, lead-column NaN fraction ~0.96% -- the tail `max(LAG_HOURS)`=672h of each series, symmetric to how the existing backward lags NaN the *head* 672h. ~55s/fit.)

## Data-leakage checks

Two distinct questions:

**1. Do the new lead features leak the FCH4 target itself?** No -- `swc_f{lag}`/`ts_f{lag}` are built only from the external per-catchment soil moisture/temperature series (`cfg()['swc']`/`cfg()['ts']`), never from FCH4 or any FCH4-derived column. Asserted programmatically below.

**2. Does any held-out (masked) timestamp leak into that same fit's training partition?** Applies identically to the backward lags already in production, but asserted as a permanent runtime guard inside `run_rf` (every fit, not a one-off check) since we're now trusting it across 3x as many feature variants. `trd = base.drop(index=gt, ...)` removes held-out timestamps *before* the pooled concat (which resets the index) -- asserted immediately after.

**Separate, non-leakage scope caveat (documented, not a bug):** the forward lag features use soil-sensor readings from *after* the timestamp being reconstructed. Legitimate for this *gap-filling* task (applied retrospectively to an already-recorded archive, those future readings genuinely exist by the time gap-filling runs -- the same justification underlying `reddyproc_pipeline.py`'s own bidirectional windows). **Not** legitimate in a live forecasting deployment -- Arms B/C are gap-filling-only findings, not transferable to the forecasting pipeline without re-deriving them as backward-only.

In [6]:
# (1) feature-purity check -- no arm's feature list references the target or FCH4 directly
for arm, spec in ARMS.items():
    bad = [c for c in spec["feat"] if c == "target" or "FCH4" in c]
    assert not bad, f"{arm}: target-derived column(s) leaked into feature list: {bad}"
print("[OK] no arm's feature list references the target/FCH4 directly")

c4 = cfg(4, ts_col_for(4))
print("lead/lag source columns (Tower 4):", c4["swc"], "|", c4["ts"])

[OK] no arm's feature list references the target/FCH4 directly
lead/lag source columns (Tower 4): Soil Moisture @ 10cm Depth (%) [Catchment 4 After  2013/08/13] | Soil Temperature @ 15cm Depth (oC) [Catchment 4 After  2013/08/13]


## Phase 2 -- full run: 3 arms x 3 towers x 5 scenarios x 2 reps (90 fits)

`run_rf(arm, t)` mirrors F-08's `run_rf(t, pooled, variant)` shape (pooled is always `True` here) but dispatches on `ARMS[arm]`, with the leakage assertion from above on every fit. **Split into one cell per (arm, tower) pair** (not one big loop) so a checkpointing driver can save the notebook to disk after each cell completes.

In [7]:
def run_rf(arm, t):
    spec = ARMS[arm]; feat = spec["feat"] + DUM
    frames = {tt: spec["frame_fn"](tt, True, DF) for tt in TOWERS}
    g = frames[t]; dm = dom_mask(g.index, t); out = {}
    for sc, gh in SCENARIOS.items():
        rr = []
        for gt in insert_calendar_gaps(g, "target", dm, gh):
            if len(gt) < 5: continue
            base = g[dm & g["target"].notna().values]; trd = base.drop(index=gt, errors="ignore")
            assert not pd.DatetimeIndex(gt).isin(trd.index).any(), (
                f"leakage: held-out timestamps present in training index "
                f"(arm={arm}, tower={t}, scenario={sc})")
            others = [frames[tt][dom_mask(frames[tt].index, tt) & frames[tt]["target"].notna().values]
                      for tt in TOWERS if tt != t]
            trd = pd.concat([trd] + others, ignore_index=True)
            rf, imp = fit(feat, trd); yp = rf.predict(imp.transform(g.loc[gt, feat].values))
            rr.append(mets(g.loc[gt, "target"].values, yp))
        out[sc] = med_metrics(rr)
    return out


RESU = {}
t0_full = time.time()

In [8]:
ta = time.time()
RESU[("baseline", 2)] = run_rf("baseline", 2)
print("done baseline Tower 2", f"({time.time()-ta:.0f}s this pair, {time.time()-t0_full:.0f}s total elapsed)", flush=True)

done baseline Tower 2 (546s this pair, 546s total elapsed)


In [9]:
ta = time.time()
RESU[("baseline", 4)] = run_rf("baseline", 4)
print("done baseline Tower 4", f"({time.time()-ta:.0f}s this pair, {time.time()-t0_full:.0f}s total elapsed)", flush=True)

done baseline Tower 4 (476s this pair, 1023s total elapsed)


In [10]:
ta = time.time()
RESU[("baseline", 9)] = run_rf("baseline", 9)
print("done baseline Tower 9", f"({time.time()-ta:.0f}s this pair, {time.time()-t0_full:.0f}s total elapsed)", flush=True)

done baseline Tower 9 (499s this pair, 1522s total elapsed)


In [11]:
ta = time.time()
RESU[("bidir", 2)] = run_rf("bidir", 2)
print("done bidir Tower 2", f"({time.time()-ta:.0f}s this pair, {time.time()-t0_full:.0f}s total elapsed)", flush=True)

done bidir Tower 2 (659s this pair, 2181s total elapsed)


In [12]:
ta = time.time()
RESU[("bidir", 4)] = run_rf("bidir", 4)
print("done bidir Tower 4", f"({time.time()-ta:.0f}s this pair, {time.time()-t0_full:.0f}s total elapsed)", flush=True)

done bidir Tower 4 (598s this pair, 2779s total elapsed)


In [13]:
ta = time.time()
RESU[("bidir", 9)] = run_rf("bidir", 9)
print("done bidir Tower 9", f"({time.time()-ta:.0f}s this pair, {time.time()-t0_full:.0f}s total elapsed)", flush=True)

done bidir Tower 9 (632s this pair, 3411s total elapsed)


In [14]:
ta = time.time()
RESU[("leadonly", 2)] = run_rf("leadonly", 2)
print("done leadonly Tower 2", f"({time.time()-ta:.0f}s this pair, {time.time()-t0_full:.0f}s total elapsed)", flush=True)

done leadonly Tower 2 (525s this pair, 3936s total elapsed)


In [15]:
ta = time.time()
RESU[("leadonly", 4)] = run_rf("leadonly", 4)
print("done leadonly Tower 4", f"({time.time()-ta:.0f}s this pair, {time.time()-t0_full:.0f}s total elapsed)", flush=True)

done leadonly Tower 4 (468s this pair, 4404s total elapsed)


In [16]:
ta = time.time()
RESU[("leadonly", 9)] = run_rf("leadonly", 9)
print("done leadonly Tower 9", f"({time.time()-ta:.0f}s this pair, {time.time()-t0_full:.0f}s total elapsed)", flush=True)

done leadonly Tower 9 (505s this pair, 4909s total elapsed)


In [17]:
print(f"Phase 2 total wall time: {time.time()-t0_full:.1f}s ({(time.time()-t0_full)/3600:.2f}h)")

Phase 2 total wall time: 4909.3s (1.36h)


## Phase 3 -- results, comparison, save

In [18]:
rows = []
for (arm, t), scn in RESU.items():
    ov = {k: np.nanmedian([scn[s][k] for s in SCENARIOS]) for k in ["R2", "RMSE", "MAE", "MBE"]}
    rows.append({"variant": arm, "tower": t, "model": "RFm_pool",
                 **{f"{k}_overall": round(ov[k], 3) for k in ov},
                 **{f"R2_{s}": round(scn[s]["R2"], 3) for s in SCENARIOS}})
R = pd.DataFrame(rows)
piv = R.pivot_table(index="tower", columns="variant", values="R2_overall")[["baseline", "bidir", "leadonly"]]
piv["d(bidir-baseline)"] = (piv["bidir"] - piv["baseline"]).round(3)
piv["d(leadonly-baseline)"] = (piv["leadonly"] - piv["baseline"]).round(3)
piv["d(bidir-leadonly)"] = (piv["bidir"] - piv["leadonly"]).round(3)
print("=== Overall median R2 by arm, + pairwise deltas ===")
print(piv.round(3).to_string())

RFM_CHAMPION = {2: 0.574, 4: 0.402, 9: 0.418}  # results/f09a_summary.csv, BEST_RESULTS.md §1
champ = piv[["baseline", "bidir", "leadonly"]].copy()
champ["champion"] = champ.index.map(RFM_CHAMPION)
for a in ["baseline", "bidir", "leadonly"]:
    champ[f"beats_champion({a})"] = champ[a] > champ["champion"]
print("\n=== vs. recorded RFm champion ===")
print(champ.round(3).to_string())

RESULTS.mkdir(exist_ok=True)
R.to_csv(RESULTS / "f12_summary.csv", index=False)
print("\nsaved results/f12_summary.csv")

=== Overall median R2 by arm, + pairwise deltas ===
variant  baseline  bidir  leadonly  d(bidir-baseline)  d(leadonly-baseline)  d(bidir-leadonly)
tower                                                                                         
2           0.574  0.555     0.557             -0.019                -0.017             -0.002
4           0.402  0.410     0.410              0.008                 0.008              0.000
9           0.418  0.408     0.407             -0.010                -0.011              0.001

=== vs. recorded RFm champion ===
variant  baseline  bidir  leadonly  champion  beats_champion(baseline)  beats_champion(bidir)  beats_champion(leadonly)
tower                                                                                                                  
2           0.574  0.555     0.557     0.574                     False                  False                     False
4           0.402  0.410     0.410     0.402                     False        

## Phase 4 -- feature-importance diagnostic (cheap add-on, single refit per arm)

In [19]:
imp_rows = []
for arm, spec in ARMS.items():
    feat = spec["feat"] + DUM
    frames = {tt: spec["frame_fn"](tt, True, DF) for tt in TOWERS}
    parts = [frames[tt][dom_mask(frames[tt].index, tt) & frames[tt]["target"].notna().values] for tt in TOWERS]
    trd_full = pd.concat(parts, ignore_index=True)
    rf, _ = fit(feat, trd_full)
    fi = native_importance_tree(rf, feat)
    lead_mass = fi.reindex(LEAD_COLS).fillna(0).sum()
    lag_mass = fi.reindex(LAG_COLS).fillna(0).sum()
    imp_rows.append({"arm": arm, "lead_importance": lead_mass, "lag_importance": lag_mass,
                      "top5": list(fi.head(5).index)})
    print(f"{arm}: lead_cols importance sum={lead_mass:.4f}, lag_cols importance sum={lag_mass:.4f}")
    print(f"  top5 features: {list(fi.head(5).index)}")
IMP = pd.DataFrame(imp_rows)
IMP

baseline: lead_cols importance sum=0.0000, lag_cols importance sum=0.1073
  top5 features: ['fc', 'lsu_dens', 'PPFD_1_1_1', 'RN_1_1_1', 'SWIN_1_1_1']


bidir: lead_cols importance sum=0.0989, lag_cols importance sum=0.0808
  top5 features: ['fc', 'lsu_dens', 'PPFD_1_1_1', 'RN_1_1_1', 'SWIN_1_1_1']


leadonly: lead_cols importance sum=0.1229, lag_cols importance sum=0.0000
  top5 features: ['fc', 'lsu_dens', 'PPFD_1_1_1', 'RN_1_1_1', 'SWIN_1_1_1']


,arm,lead_importance,lag_importance,top5
0,baseline,0.000000,0.107270,"[fc, lsu_dens, PPFD_1_1_1, RN_1_1_1, SWIN_1_1_1]"
1,bidir,0.098868,0.080794,"[fc, lsu_dens, PPFD_1_1_1, RN_1_1_1, SWIN_1_1_1]"
2,leadonly,0.122915,0.000000,"[fc, lsu_dens, PPFD_1_1_1, RN_1_1_1, SWIN_1_1_1]"


## Append to benchmarks.csv

In [20]:
bench = RESULTS / "benchmarks.csv"; today = datetime.date.today().isoformat()
ex = pd.read_csv(bench); ex = ex[ex["replication"] != "F-12"]
brows = []
for (arm, t), scn in RESU.items():
    fs = ARMS[arm]["fs_tag"]
    for sc in SCENARIOS:
        m = scn[sc]
        if pd.isna(m["R2"]) and pd.isna(m["RMSE"]): continue
        brows.append({"replication": "F-12", "model": "RFm", "tower": f"Tower {t}",
            "feature_set": fs, "scenario": sc, "split": "fullCV",
            "R2": round(float(m["R2"]), 4) if pd.notna(m["R2"]) else np.nan,
            "RMSE": round(float(m["RMSE"]), 4), "MAE": round(float(m["MAE"]), 4), "MBE": round(float(m["MBE"]), 4),
            "date": today,
            "notes": "F12 backward-vs-bidirectional-vs-leads-only soil lag ablation, partial-pooled EXT RFm; N_REPS=2 (reduced from F-08's 5, documented in notebook)"})
new = pd.DataFrame(brows); comb = pd.concat([ex, new], ignore_index=True); comb.to_csv(bench, index=False)
print(f"Wrote {len(new)} F-12 rows. Total {len(comb)}.")

Wrote 45 F-12 rows. Total 3983.
